In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (keep original integer seconds as well)
for df in (metric_df, log_df, trace_df, error_df):
    if 'timestamp' in df.columns:
        # Ensure timestamp is integer-like
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        # Create UTC datetime column
        df['ts'] = pd.to_datetime(df['timestamp'].astype('Int64'), unit='s', utc=True)

# Helper to get cmdb counts (top N)
def top_cmdb_counts(df, topn=50):
    if 'cmdb_id' not in df.columns or df['cmdb_id'].dropna().empty:
        return []
    vc = df['cmdb_id'].value_counts().reset_index()
    vc.columns = ['cmdb_id', 'count']
    return vc.head(topn).to_dict('records')

# Metric summary
metric_cmdb_counts = top_cmdb_counts(metric_df, topn=50)
metric_kpis = list(pd.Series(metric_df['kpi_name'].dropna().unique())[:50])
metric_min_ts = int(metric_df['timestamp'].min()) if not metric_df['timestamp'].isna().all() else None
metric_max_ts = int(metric_df['timestamp'].max()) if not metric_df['timestamp'].isna().all() else None
metric_row_count = int(len(metric_df))

metric_result = {
    "cmdb_counts_top50": metric_cmdb_counts,
    "kpi_names_top50": metric_kpis,
    "kpi_names_total_unique": int(metric_df['kpi_name'].nunique(dropna=True)),
    "min_timestamp": metric_min_ts,
    "max_timestamp": metric_max_ts,
    "row_count": metric_row_count
}

# Log summary
log_cmdb_counts = top_cmdb_counts(log_df, topn=50)
log_names = list(pd.Series(log_df['log_name'].dropna().unique())[:50])
log_min_ts = int(log_df['timestamp'].min()) if not log_df['timestamp'].isna().all() else None
log_max_ts = int(log_df['timestamp'].max()) if not log_df['timestamp'].isna().all() else None
log_row_count = int(len(log_df))

log_result = {
    "cmdb_counts_top50": log_cmdb_counts,
    "log_names_top50": log_names,
    "log_names_total_unique": int(log_df['log_name'].nunique(dropna=True)),
    "min_timestamp": log_min_ts,
    "max_timestamp": log_max_ts,
    "row_count": log_row_count
}

# Trace summary
trace_cmdb_counts = top_cmdb_counts(trace_df, topn=50)
trace_names = list(pd.Series(trace_df['trace_name'].dropna().unique())[:50])
trace_min_ts = int(trace_df['timestamp'].min()) if not trace_df['timestamp'].isna().all() else None
trace_max_ts = int(trace_df['timestamp'].max()) if not trace_df['timestamp'].isna().all() else None
trace_row_count = int(len(trace_df))

trace_result = {
    "cmdb_counts_top50": trace_cmdb_counts,
    "trace_names_top50": trace_names,
    "trace_names_total_unique": int(trace_df['trace_name'].nunique(dropna=True)),
    "min_timestamp": trace_min_ts,
    "max_timestamp": trace_max_ts,
    "row_count": trace_row_count
}

# Error logs summary: sample up to 10 rows with truncated message (first 200 chars)
err_row_count = int(len(error_df))
err_min_ts = int(error_df['timestamp'].min()) if not error_df['timestamp'].isna().all() else None
err_max_ts = int(error_df['timestamp'].max()) if not error_df['timestamp'].isna().all() else None

# Prepare sample (head up to 10). Truncate message.
if {'timestamp', 'cmdb_id', 'message'}.issubset(error_df.columns):
    sample_df = error_df[['timestamp', 'cmdb_id', 'message']].head(10).copy()
    sample_df['message'] = sample_df['message'].astype(str).str.slice(0, 200)
    error_sample = sample_df.to_dict('records')
else:
    error_sample = []

error_result = {
    "sample_up_to_10": error_sample,
    "min_timestamp": err_min_ts,
    "max_timestamp": err_max_ts,
    "row_count": err_row_count
}

# Combine results into a single JSON-like dict (compact)
result = {
    "metric.csv": metric_result,
    "log.csv": log_result,
    "trace.csv": trace_result,
    "error_logs.csv": error_result
}

result

```
Out[1]:
```
Summary of the loaded telemetry (plain English):

- metric.csv
  - Rows: 2,050
  - Time range (unix seconds UTC): min=1705653900, max=1705655340
  - Unique KPI names (8 total, top list): cpu, error, latency-50, latency-90, mem, socket, workload, diskio
  - Top cmdb_id by row count (top entries): catalogue (200), user (200), payment (200), orders (175), front-end (175), carts (175), shipping (150), several DBs and infrastructure entries at 100, rabbitmq-exporter (75)

- log.csv
  - Rows: 460
  - Time range (unix seconds UTC): min=1705653900, max=1705655340
  - Unique log names (2 total): log.error_count, log.row_count
  - Top cmdb_id by row count: many services at 50 rows (carts-db, catalogue, front-end, orders, payment, queue-master, shipping, user), carts (46), session-db (10), user-db (4)

- trace.csv
  - No data (0 rows). No trace_names, no cmdb counts, no timestamps.

- error_logs.csv
  - No data (0 rows). No sample rows, no timestamps.

Implication: metric and log telemetry are available across most services for the given window and can be used for diagnosis; trace and error_log files contain no records to inspect.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'cmdb_counts_top50': [{'cmdb_id': 'catalogue', 'count': 200}, {'cmdb_id': 'user', 'count': 200}, {'cmdb_id': 'payment', 'count': 200}, {'cmdb_id': 'orders', 'count': 175}, {'cmdb_id': 'front-end', 'count': 175}, {'cmdb_id': 'carts', 'count': 175}, {'cmdb_id': 'shipping', 'count': 150}, {'cmdb_id': 'orders-db', 'count': 100}, {'cmdb_id': 'catalogue-db', 'count': 100}, {'cmdb_id': 'carts-db', 'count': 100}, {'cmdb_id': 'queue-master', 'count': 100}, {'cmdb_id': 'session-db', 'count': 100}, {'cmdb_id': 'rabbitmq', 'count': 100}, {'cmdb_id': 'user-db', 'count': 100}, {'cmdb_id': 'rabbitmq-exporter', 'count': 75}], 'kpi_names_top50': ['cpu', 'error', 'latency-50', 'latency-90', 'mem', 'socket', 'workload', 'diskio'], 'kpi_names_total_unique': 8, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'row_count': 2050}, 'log.csv': {'cmdb_counts_top50': [{'cmdb_id': 'carts-db', 'count': 50}, {'cmdb_id': 'catalogue', 'count': 50}, {'cmdb_id': 'front-end', 'count': 50}, {'cmdb_id': 'orders', 'count': 50}, {'cmdb_id': 'payment', 'count': 50}, {'cmdb_id': 'queue-master', 'count': 50}, {'cmdb_id': 'shipping', 'count': 50}, {'cmdb_id': 'user', 'count': 50}, {'cmdb_id': 'carts', 'count': 46}, {'cmdb_id': 'session-db', 'count': 10}, {'cmdb_id': 'user-db', 'count': 4}], 'log_names_top50': ['log.error_count', 'log.row_count'], 'log_names_total_unique': 2, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'row_count': 460}, 'trace.csv': {'cmdb_counts_top50': [], 'trace_names_top50': [], 'trace_names_total_unique': 0, 'min_timestamp': None, 'max_timestamp': None, 'row_count': 0}, 'error_logs.csv': {'sample_up_to_10': [], 'min_timestamp': None, 'max_timestamp': None, 'row_count': 0}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

# Ensure timestamp numeric ints and created ts columns (UTC)
for df in (metric_df, log_df):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['ts'] = pd.to_datetime(df['timestamp'].astype('Int64'), unit='s', utc=True)

# Candidate services and incident window (UTC)
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']
start_iso = '2024-01-19T08:42:16Z'
end_iso =   '2024-01-19T09:12:16Z'
start_ts = int(pd.to_datetime(start_iso, utc=True).timestamp())
end_ts = int(pd.to_datetime(end_iso, utc=True).timestamp())

# --- Metric analysis ---
metric_rows = []
# Group by cmdb_id and kpi_name across full metric_df
grp = metric_df.dropna(subset=['cmdb_id','kpi_name']).groupby(['cmdb_id','kpi_name'])
for (cmdb, kpi), g in grp:
    if cmdb not in candidates:
        continue
    values = g['value'].dropna().astype(float)
    if values.empty:
        continue
    p95 = float(values.quantile(0.95))
    total_points = int(len(g))
    min_ts = int(g['timestamp'].min())
    max_ts = int(g['timestamp'].max())
    # filter to incident window (inclusive)
    gw = g[(g['timestamp'] >= start_ts) & (g['timestamp'] <= end_ts)]
    count_in_window = int(len(gw))
    exceed_mask = gw['value'].astype(float) > p95
    exceed_count = int(exceed_mask.sum())
    if exceed_count > 0:
        earliest_exceed_ts = int(gw.loc[exceed_mask, 'timestamp'].min())
    else:
        earliest_exceed_ts = None
    if count_in_window > 0:
        max_value_in_window = float(gw['value'].astype(float).max())
    else:
        max_value_in_window = None
    severity_ratio = None
    if (max_value_in_window is not None) and (p95 != 0):
        severity_ratio = float(max_value_in_window / p95)
    # Round numeric outputs to 4 decimals where applicable
    entry = {
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95': round(p95, 4),
        'total_points': total_points,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'count_in_window': count_in_window,
        'exceed_count_in_window': exceed_count,
        'earliest_exceed_timestamp': earliest_exceed_ts,
        'max_value_in_window': round(max_value_in_window,4) if max_value_in_window is not None else None,
        'severity_ratio': round(severity_ratio,4) if severity_ratio is not None else None
    }
    metric_rows.append(entry)

# Sort metric_rows by exceed_count desc then severity_ratio desc, limit top20
metric_df_out = pd.DataFrame(metric_rows)
if not metric_df_out.empty:
    # For sorting, treat None severity_ratio as -inf so they go last
    metric_df_out['severity_sort'] = metric_df_out['severity_ratio'].fillna(-np.inf)
    metric_df_out = metric_df_out.sort_values(by=['exceed_count_in_window','severity_sort'], ascending=[False,False])
    metric_df_out = metric_df_out.drop(columns=['severity_sort'])
    metric_list = metric_df_out.head(20).to_dict('records')
else:
    metric_list = []

# --- Log analysis ---
log_rows = []
grp_log = log_df.dropna(subset=['cmdb_id','log_name']).groupby(['cmdb_id','log_name'])
for (cmdb, logname), g in grp_log:
    if cmdb not in candidates:
        continue
    values = g['value'].dropna().astype(float)
    if values.empty:
        continue
    p95 = float(values.quantile(0.95))
    total_points = int(len(g))
    min_ts = int(g['timestamp'].min())
    max_ts = int(g['timestamp'].max())
    # filter to incident window (inclusive)
    gw = g[(g['timestamp'] >= start_ts) & (g['timestamp'] <= end_ts)]
    count_in_window = int(len(gw))
    exceed_mask = gw['value'].astype(float) > p95
    exceed_count = int(exceed_mask.sum())
    if exceed_count > 0:
        earliest_exceed_ts = int(gw.loc[exceed_mask, 'timestamp'].min())
    else:
        earliest_exceed_ts = None
    if count_in_window > 0:
        max_value_in_window = float(gw['value'].astype(float).max())
    else:
        max_value_in_window = None
    severity_ratio = None
    if (max_value_in_window is not None) and (p95 != 0):
        severity_ratio = float(max_value_in_window / p95)
    entry = {
        'cmdb_id': cmdb,
        'log_name': logname,
        'p95': round(p95,4),
        'total_points': total_points,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'count_in_window': count_in_window,
        'exceed_count_in_window': exceed_count,
        'earliest_exceed_timestamp': earliest_exceed_ts,
        'max_value_in_window': round(max_value_in_window,4) if max_value_in_window is not None else None,
        'severity_ratio': round(severity_ratio,4) if severity_ratio is not None else None
    }
    log_rows.append(entry)

# Sort log_rows by exceed_count desc then severity_ratio desc, limit top20
log_df_out = pd.DataFrame(log_rows)
if not log_df_out.empty:
    log_df_out['severity_sort'] = log_df_out['severity_ratio'].fillna(-np.inf)
    log_df_out = log_df_out.sort_values(by=['exceed_count_in_window','severity_sort'], ascending=[False,False])
    log_df_out = log_df_out.drop(columns=['severity_sort'])
    log_list = log_df_out.head(20).to_dict('records')
else:
    log_list = []

# --- Summary: which components had any anomalies in window ---
metric_anomaly_components = sorted(
    set([r['cmdb_id'] for r in metric_list if r['exceed_count_in_window'] and r['exceed_count_in_window']>0])
)
log_anomaly_components = sorted(
    set([r['cmdb_id'] for r in log_list if r['exceed_count_in_window'] and r['exceed_count_in_window']>0])
)

# Also note components that had no data at all in metrics/logs
components_no_metric_data = [c for c in candidates if not any((m['cmdb_id']==c) for m in metric_rows)]
components_no_log_data = [c for c in candidates if not any((l['cmdb_id']==c) for l in log_rows)]

summary = {
    'metric_anomaly_components': metric_anomaly_components,
    'log_anomaly_components': log_anomaly_components,
    'components_with_no_metric_data': components_no_metric_data,
    'components_with_no_log_data': components_no_log_data
}

final_result = {
    'metric_analysis_top20': metric_list,
    'log_analysis_top20': log_list,
    'summary': summary
}

final_result

```
Out[2]:
```
summary = (
    "Summary of findings (plain English):\n"
    "- All five candidate services (carts, catalogue, orders, payment, user) show metric anomalies during the incident window: each service had at least one KPI with points exceeding its global P95. "
    "Examples of impacted KPIs include carts (cpu, latency-90, latency-50, socket, workload), orders (latency-90, latency-50, cpu, workload), payment (latency-90, latency-50, cpu, mem, workload), user (socket, latency-50, latency-90, mem), and catalogue (workload, cpu).\n"
    "- Log anomalies: log.row_count exceeded its global P95 in all five services (each had 1–2 exceedances in-window). log.error_count remained 0 for these services (no error spikes).\n"
    "- Notable severity values: highest metric severity in the results was carts cpu (severity_ratio ≈ 1.1605) and carts latency-90 (≈ 1.104). The largest log severity was carts log.row_count (severity_ratio ≈ 1.384).\n"
    "- Data coverage: none of the five components lacked metric or log data for the analyzed period.\n"
    "Overall: metrics and row-count logs indicate mild-but-widespread anomalies across all five services during the window, with carts showing the strongest signal."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_analysis_top20': [{'cmdb_id': 'carts', 'kpi_name': 'cpu', 'p95': 1.5934, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654620.0, 'max_value_in_window': 1.8492, 'severity_ratio': 1.1605}, {'cmdb_id': 'carts', 'kpi_name': 'latency-90', 'p95': 8.0153, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654500.0, 'max_value_in_window': 8.8486, 'severity_ratio': 1.104}, {'cmdb_id': 'payment', 'kpi_name': 'latency-90', 'p95': 0.0064, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705655280.0, 'max_value_in_window': 0.007, 'severity_ratio': 1.099}, {'cmdb_id': 'carts', 'kpi_name': 'latency-50', 'p95': 3.1499, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654500.0, 'max_value_in_window': 3.3187, 'severity_ratio': 1.0536}, {'cmdb_id': 'orders', 'kpi_name': 'latency-90', 'p95': 2.7461, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654560.0, 'max_value_in_window': 2.8826, 'severity_ratio': 1.0497}, {'cmdb_id': 'user', 'kpi_name': 'socket', 'p95': 41.1533, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705655280.0, 'max_value_in_window': 43.0, 'severity_ratio': 1.0449}, {'cmdb_id': 'payment', 'kpi_name': 'latency-50', 'p95': 0.0031, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654860.0, 'max_value_in_window': 0.0032, 'severity_ratio': 1.0444}, {'cmdb_id': 'orders', 'kpi_name': 'latency-50', 'p95': 1.448, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705655220.0, 'max_value_in_window': 1.5032, 'severity_ratio': 1.0382}, {'cmdb_id': 'carts', 'kpi_name': 'socket', 'p95': 60.4379, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654500.0, 'max_value_in_window': 61.8, 'severity_ratio': 1.0225}, {'cmdb_id': 'user', 'kpi_name': 'latency-50', 'p95': 0.0843, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654860.0, 'max_value_in_window': 0.0861, 'severity_ratio': 1.0211}, {'cmdb_id': 'catalogue', 'kpi_name': 'workload', 'p95': 3.5125, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654140.0, 'max_value_in_window': 3.5812, 'severity_ratio': 1.0195}, {'cmdb_id': 'payment', 'kpi_name': 'workload', 'p95': 1.7945, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654680.0, 'max_value_in_window': 1.8244, 'severity_ratio': 1.0167}, {'cmdb_id': 'catalogue', 'kpi_name': 'cpu', 'p95': 0.1516, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654260.0, 'max_value_in_window': 0.154, 'severity_ratio': 1.0158}, {'cmdb_id': 'user', 'kpi_name': 'mem', 'p95': 16028753.92, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654740.0, 'max_value_in_window': 16268902.4, 'severity_ratio': 1.015}, {'cmdb_id': 'payment', 'kpi_name': 'mem', 'p95': 4482371.6643, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654380.0, 'max_value_in_window': 4532224.0, 'severity_ratio': 1.0111}, {'cmdb_id': 'payment', 'kpi_name': 'cpu', 'p95': 0.078, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654140.0, 'max_value_in_window': 0.0787, 'severity_ratio': 1.0096}, {'cmdb_id': 'orders', 'kpi_name': 'workload', 'p95': 1.8092, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654680.0, 'max_value_in_window': 1.8256, 'severity_ratio': 1.0091}, {'cmdb_id': 'orders', 'kpi_name': 'cpu', 'p95': 0.9795, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654620.0, 'max_value_in_window': 0.9879, 'severity_ratio': 1.0086}, {'cmdb_id': 'user', 'kpi_name': 'latency-90', 'p95': 0.2132, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654980.0, 'max_value_in_window': 0.2147, 'severity_ratio': 1.0071}, {'cmdb_id': 'carts', 'kpi_name': 'workload', 'p95': 7.0174, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705654140.0, 'max_value_in_window': 7.0611, 'severity_ratio': 1.0062}], 'log_analysis_top20': [{'cmdb_id': 'user', 'log_name': 'log.row_count', 'p95': 524.4, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 2, 'earliest_exceed_timestamp': 1705653960.0, 'max_value_in_window': 528.0, 'severity_ratio': 1.0069}, {'cmdb_id': 'carts', 'log_name': 'log.row_count', 'p95': 125.0, 'total_points': 23, 'min_timestamp': 1705653960, 'max_timestamp': 1705655340, 'count_in_window': 23, 'exceed_count_in_window': 1, 'earliest_exceed_timestamp': 1705654560.0, 'max_value_in_window': 173.0, 'severity_ratio': 1.384}, {'cmdb_id': 'orders', 'log_name': 'log.row_count', 'p95': 110.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 1, 'earliest_exceed_timestamp': 1705655040.0, 'max_value_in_window': 114.0, 'severity_ratio': 1.0364}, {'cmdb_id': 'catalogue', 'log_name': 'log.row_count', 'p95': 144.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 1, 'earliest_exceed_timestamp': 1705654680.0, 'max_value_in_window': 148.0, 'severity_ratio': 1.0278}, {'cmdb_id': 'payment', 'log_name': 'log.row_count', 'p95': 95.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 1, 'earliest_exceed_timestamp': 1705655040.0, 'max_value_in_window': 97.0, 'severity_ratio': 1.0211}, {'cmdb_id': 'carts', 'log_name': 'log.error_count', 'p95': 0.0, 'total_points': 23, 'min_timestamp': 1705653960, 'max_timestamp': 1705655340, 'count_in_window': 23, 'exceed_count_in_window': 0, 'earliest_exceed_timestamp': nan, 'max_value_in_window': 0.0, 'severity_ratio': nan}, {'cmdb_id': 'catalogue', 'log_name': 'log.error_count', 'p95': 0.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 0, 'earliest_exceed_timestamp': nan, 'max_value_in_window': 0.0, 'severity_ratio': nan}, {'cmdb_id': 'orders', 'log_name': 'log.error_count', 'p95': 0.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 0, 'earliest_exceed_timestamp': nan, 'max_value_in_window': 0.0, 'severity_ratio': nan}, {'cmdb_id': 'payment', 'log_name': 'log.error_count', 'p95': 0.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 0, 'earliest_exceed_timestamp': nan, 'max_value_in_window': 0.0, 'severity_ratio': nan}, {'cmdb_id': 'user', 'log_name': 'log.error_count', 'p95': 0.0, 'total_points': 25, 'min_timestamp': 1705653900, 'max_timestamp': 1705655340, 'count_in_window': 25, 'exceed_count_in_window': 0, 'earliest_exceed_timestamp': nan, 'max_value_in_window': 0.0, 'severity_ratio': nan}], 'summary': {'metric_anomaly_components': ['carts', 'catalogue', 'orders', 'payment', 'user'], 'log_anomaly_components': ['carts', 'catalogue', 'orders', 'payment', 'user'], 'components_with_no_metric_data': [], 'components_with_no_log_data': []}}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

# Ensure timestamp numeric ints and UTC ts column
for df in (metric_df, log_df):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['ts'] = pd.to_datetime(df['timestamp'].astype('Int64'), unit='s', utc=True)

# Focus settings
component = 'carts'
metric_kpis_requested = ['cpu','latency-90','latency-50','socket','workload','mem']
log_names_requested = ['log.row_count','log.error_count']
start_iso = '2024-01-19T08:42:16Z'
end_iso   = '2024-01-19T09:12:16Z'
start_ts = int(pd.to_datetime(start_iso, utc=True).timestamp())
end_ts = int(pd.to_datetime(end_iso, utc=True).timestamp())

def analyze_series_full_then_window(df, group_cols, value_col, group_values, window_start, window_end,
                                     list_limit=30):
    """
    df: dataframe containing group_cols + value_col + timestamp
    group_cols: list of grouping column names (e.g., ['cmdb_id','kpi_name'] or ['cmdb_id','log_name'])
    group_values: tuple/list values matching group_cols to select (e.g., ('carts','cpu'))
    Returns dict with p95 and window analysis
    """
    selector = True
    for col, val in zip(group_cols, group_values):
        selector &= (df[col] == val)
    gfull = df.loc[selector].copy()
    if gfull.empty:
        return None  # missing
    # compute global p95 on full series (rule: compute before filtering)
    vals_full = gfull[value_col].dropna().astype(float)
    p95 = float(vals_full.quantile(0.95))
    # window slice (inclusive)
    gwin = gfull[(gfull['timestamp'] >= window_start) & (gfull['timestamp'] <= window_end)].copy()
    # sort window by timestamp asc
    if not gwin.empty:
        gwin = gwin.sort_values('timestamp').reset_index(drop=True)
    # prepare list of rows in-window limited to list_limit
    rows_list = []
    for _, r in gwin.head(list_limit).iterrows():
        v = None if pd.isna(r[value_col]) else float(r[value_col])
        rows_list.append({
            'timestamp': int(r['timestamp']),
            'value': round(v,4) if v is not None else None,
            'exceeds_p95': (v is not None) and (v > p95)
        })
    # earliest exceed timestamp in window
    if not gwin.empty:
        exceeded = gwin[gwin[value_col].astype(float) > p95]
        if not exceeded.empty:
            earliest_exceed_ts = int(exceeded['timestamp'].min())
        else:
            earliest_exceed_ts = None
    else:
        earliest_exceed_ts = None
    # compute count_of_consecutive_exceedances_at_the_start_of_fault:
    # Find the index of the earliest exceed in the window (in the sorted gwin). Then expand left/right to count contiguous True values.
    consecutive_count = 0
    if earliest_exceed_ts is not None:
        # boolean series of exceedances aligned with gwin
        exceed_bool = gwin[value_col].astype(float) > p95
        # find position(s) where timestamp == earliest_exceed_ts (take first)
        pos_series = gwin.index[gwin['timestamp'] == earliest_exceed_ts].tolist()
        if pos_series:
            pos = pos_series[0]
            # expand left
            left = pos
            while left-1 >= 0 and exceed_bool.iat[left-1]:
                left -= 1
            # expand right
            right = pos
            while right+1 < len(exceed_bool) and exceed_bool.iat[right+1]:
                right += 1
            consecutive_count = int(right - left + 1)
        else:
            consecutive_count = 0
    else:
        consecutive_count = 0
    # max value in window
    if not gwin.empty:
        max_value_in_window = float(gwin[value_col].astype(float).max())
    else:
        max_value_in_window = None
    severity_ratio = None
    if (max_value_in_window is not None) and (p95 != 0):
        severity_ratio = float(max_value_in_window / p95)
    # Build result
    res = {
        'p95': round(p95,4),
        'rows_in_window_limited': rows_list,
        'earliest_exceed_timestamp': int(earliest_exceed_ts) if earliest_exceed_ts is not None else None,
        'consecutive_exceedances_at_earliest': consecutive_count,
        'max_value_in_window': round(max_value_in_window,4) if max_value_in_window is not None else None,
        'severity_ratio': round(severity_ratio,4) if severity_ratio is not None else None,
        'total_points_full_series': int(len(gfull)),
        'min_timestamp_full_series': int(gfull['timestamp'].min()),
        'max_timestamp_full_series': int(gfull['timestamp'].max())
    }
    return res

# Metric analysis for carts
metric_results = {}
for kpi in metric_kpis_requested:
    out = analyze_series_full_then_window(
        metric_df,
        group_cols=['cmdb_id','kpi_name'],
        value_col='value',
        group_values=(component, kpi),
        window_start=start_ts,
        window_end=end_ts,
        list_limit=30
    )
    if out is None:
        metric_results[kpi] = {'note': f"missing for {component}"}
    else:
        metric_results[kpi] = out

# Log analysis for carts
log_results = {}
for lname in log_names_requested:
    out = analyze_series_full_then_window(
        log_df,
        group_cols=['cmdb_id','log_name'],
        value_col='value',
        group_values=(component, lname),
        window_start=start_ts,
        window_end=end_ts,
        list_limit=30
    )
    if out is None:
        log_results[lname] = {'note': f"missing for {component}"}
    else:
        log_results[lname] = out

final_result = {
    'component': component,
    'incident_window': {'start': start_iso, 'end': end_iso},
    'metric_analysis': metric_results,
    'log_analysis': log_results
}

final_result

```
Out[3]:
```
Summary for component "carts" (plain English):

- Conclusion: carts shows a confirmed fault signal during the incident window — multiple metrics exceeded their global P95 and the row-count log has a clear spike. cpu and latency are the strongest signals.

- Metric highlights (p95, earliest exceed timestamp (UTC epoch sec), consecutive exceedances at that start, max_in_window, severity = max / p95):
  - cpu: p95=1.5934; earliest_exceed=1705654620; consecutive=2; max=1.8492; severity=1.1605
  - latency-90: p95=8.0153; earliest_exceed=1705654500; consecutive=2; max=8.8486; severity=1.1040
  - latency-50: p95=3.1499; earliest_exceed=1705654500; consecutive=1; max=3.3187; severity=1.0536
  - socket: p95=60.4379; earliest_exceed=1705654500; consecutive=2; max=61.8; severity=1.0225
  - workload: p95=7.0174; earliest_exceed=1705654140; consecutive=1; max=7.0611; severity=1.0062
  - mem: p95=223718285.9545; earliest_exceed=1705655100; consecutive=1; max=223756356.2667; severity=1.0002

- Log highlights:
  - log.row_count: p95=125.0; earliest_exceed=1705654560; consecutive=1; max=173.0; severity=1.3840 (notable spike)
  - log.error_count: p95=0.0; no exceedances (max=0.0)

- Additional notes:
  - The metric exceedances are not huge multiples of baseline (severity ratios mostly ~1.01–1.16), but they are consistent across multiple KPIs and coincide with a strong row_count spike (severity 1.384). 
  - Earliest signals for latency and socket occur around timestamp 1705654500 and cpu around 1705654620 (all within the incident window).
  - log.error_count shows no error spikes, so the issue appears performance/resource-related rather than error-rate related.

Short verdict: carts exhibits multi-metric performance anomalies during the incident window, with cpu and latency metrics and a pronounced log.row_count spike — treat carts as a confirmed candidate for the fault.

The original code execution output of IPython Kernel is also provided below for reference:

{'component': 'carts', 'incident_window': {'start': '2024-01-19T08:42:16Z', 'end': '2024-01-19T09:12:16Z'}, 'metric_analysis': {'cpu': {'p95': 1.5934, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 1.2289, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 1.2442, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 1.3085, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 1.2967, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 1.3391, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 1.3551, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 1.322, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 1.2867, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 1.3344, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 1.2559, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 1.2084, 'exceeds_p95': False}, {'timestamp': 1705654560, 'value': 1.4564, 'exceeds_p95': False}, {'timestamp': 1705654620, 'value': 1.6277, 'exceeds_p95': True}, {'timestamp': 1705654680, 'value': 1.8492, 'exceeds_p95': True}, {'timestamp': 1705654740, 'value': 1.1895, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 1.2248, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 1.193, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 1.2072, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 1.2338, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 1.1646, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 1.1034, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 1.2172, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 1.261, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 1.2494, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 1.1265, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654620, 'consecutive_exceedances_at_earliest': 2, 'max_value_in_window': 1.8492, 'severity_ratio': 1.1605, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}, 'latency-90': {'p95': 8.0153, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 6.4813, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 6.8014, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 6.2059, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 6.5535, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 7.1359, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 6.6424, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 6.4738, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 6.5957, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 5.0057, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 4.7008, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 8.8486, 'exceeds_p95': True}, {'timestamp': 1705654560, 'value': 8.0886, 'exceeds_p95': True}, {'timestamp': 1705654620, 'value': 7.6568, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 6.9712, 'exceeds_p95': False}, {'timestamp': 1705654740, 'value': 6.9313, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 5.3652, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 7.6296, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 6.3126, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 5.0779, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 7.3605, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 5.5515, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 7.1817, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 7.7221, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 7.6786, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 7.2451, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654500, 'consecutive_exceedances_at_earliest': 2, 'max_value_in_window': 8.8486, 'severity_ratio': 1.104, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}, 'latency-50': {'p95': 3.1499, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 2.1278, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 2.301, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 2.1928, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 2.1624, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 2.3016, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 2.146, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 2.3874, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 2.5209, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 1.7838, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 1.5697, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 3.3187, 'exceeds_p95': True}, {'timestamp': 1705654560, 'value': 2.6325, 'exceeds_p95': False}, {'timestamp': 1705654620, 'value': 2.0097, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 2.2735, 'exceeds_p95': False}, {'timestamp': 1705654740, 'value': 2.2333, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 1.4552, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 2.9024, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 2.5367, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 1.3931, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 2.5028, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 2.0488, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 2.4433, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 3.2016, 'exceeds_p95': True}, {'timestamp': 1705655280, 'value': 2.9431, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 2.6499, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654500, 'consecutive_exceedances_at_earliest': 1, 'max_value_in_window': 3.3187, 'severity_ratio': 1.0536, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}, 'socket': {'p95': 60.4379, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 59.5227, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 56.9833, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 56.5667, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 56.7667, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 55.6833, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 55.55, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 54.1833, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 54.1833, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 52.4, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 54.1, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 61.8, 'exceeds_p95': True}, {'timestamp': 1705654560, 'value': 60.6667, 'exceeds_p95': True}, {'timestamp': 1705654620, 'value': 55.35, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 52.9167, 'exceeds_p95': False}, {'timestamp': 1705654740, 'value': 48.0167, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 47.9167, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 48.8, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 46.35, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 47.3667, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 49.5, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 44.0333, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 48.9, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 47.75, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 47.2667, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 51.0, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654500, 'consecutive_exceedances_at_earliest': 2, 'max_value_in_window': 61.8, 'severity_ratio': 1.0225, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}, 'workload': {'p95': 7.0174, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 6.9894, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 6.8701, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 6.6467, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 6.6222, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 7.0611, 'exceeds_p95': True}, {'timestamp': 1705654200, 'value': 6.8155, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 6.88, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 6.7591, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 6.769, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 6.5201, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 6.3556, 'exceeds_p95': False}, {'timestamp': 1705654560, 'value': 6.5032, 'exceeds_p95': False}, {'timestamp': 1705654620, 'value': 6.4456, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 6.8488, 'exceeds_p95': False}, {'timestamp': 1705654740, 'value': 6.7566, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 6.3711, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 6.9766, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 6.6746, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 6.4066, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 7.0244, 'exceeds_p95': True}, {'timestamp': 1705655100, 'value': 6.3966, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 6.7635, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 6.5856, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 6.6067, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 6.3099, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654140, 'consecutive_exceedances_at_earliest': 1, 'max_value_in_window': 7.0611, 'severity_ratio': 1.0062, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}, 'mem': {'p95': 223718285.9545, 'rows_in_window_limited': [{'timestamp': 1705653900, 'value': 223218222.5455, 'exceeds_p95': False}, {'timestamp': 1705653960, 'value': 223173017.6, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 223145574.4, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 223149943.4667, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 223147212.8, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 223174519.4667, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 223178683.7333, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 223160388.2667, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 223158272.0, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 223216230.4, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 223210222.9333, 'exceeds_p95': False}, {'timestamp': 1705654560, 'value': 223386555.7333, 'exceeds_p95': False}, {'timestamp': 1705654620, 'value': 223537356.8, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 223580706.1333, 'exceeds_p95': False}, {'timestamp': 1705654740, 'value': 223580706.1333, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 223608558.9333, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 223538380.8, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 223592652.8, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 223652864.0, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 223669452.8, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 223756356.2667, 'exceeds_p95': True}, {'timestamp': 1705655160, 'value': 223700241.0667, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 223687543.4667, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 223679214.9333, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 223722797.1765, 'exceeds_p95': True}], 'earliest_exceed_timestamp': 1705655100, 'consecutive_exceedances_at_earliest': 1, 'max_value_in_window': 223756356.2667, 'severity_ratio': 1.0002, 'total_points_full_series': 25, 'min_timestamp_full_series': 1705653900, 'max_timestamp_full_series': 1705655340}}, 'log_analysis': {'log.row_count': {'p95': 125.0, 'rows_in_window_limited': [{'timestamp': 1705653960, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 75.0, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 25.0, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 75.0, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 125.0, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 50.0, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 25.0, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 25.0, 'exceeds_p95': False}, {'timestamp': 1705654560, 'value': 173.0, 'exceeds_p95': True}, {'timestamp': 1705654620, 'value': 77.0, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 50.0, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 125.0, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 25.0, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 50.0, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 100.0, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 50.0, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 50.0, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 75.0, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 25.0, 'exceeds_p95': False}], 'earliest_exceed_timestamp': 1705654560, 'consecutive_exceedances_at_earliest': 1, 'max_value_in_window': 173.0, 'severity_ratio': 1.384, 'total_points_full_series': 23, 'min_timestamp_full_series': 1705653960, 'max_timestamp_full_series': 1705655340}, 'log.error_count': {'p95': 0.0, 'rows_in_window_limited': [{'timestamp': 1705653960, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654020, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654080, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654140, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654200, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654260, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654320, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654380, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654440, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654500, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654560, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654620, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654680, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654800, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654860, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654920, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705654980, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655040, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655100, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655160, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655220, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655280, 'value': 0.0, 'exceeds_p95': False}, {'timestamp': 1705655340, 'value': 0.0, 'exceeds_p95': False}], 'earliest_exceed_timestamp': None, 'consecutive_exceedances_at_earliest': 0, 'max_value_in_window': 0.0, 'severity_ratio': None, 'total_points_full_series': 23, 'min_timestamp_full_series': 1705653960, 'max_timestamp_full_series': 1705655340}}}```
```